In [1]:
import json
import os
import tarfile
import time
import urllib.request
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

In [2]:
# 1. Pipeline Environment Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Active Engine Backend: {device}")

🚀 Active Engine Backend: cpu


In [3]:
# Production-grade ImageNet preprocessing chain (Matches Phase 4 & Phase 5 requirements)
transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),  # Standard input for high-tier models
        transforms.ToTensor(),
        # Exact ImageNet mean & standard deviation for Transfer Learning
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),
    ]
)

In [4]:
# 2. Automated Oxford Dataset Retrieval & Assembly
url = "https://www.robots.ox.ac.uk/~vgg/data/flowers/17/17flowers.tgz"
archive_name = "flowers_oxford.tgz"
extraction_dir = "./oxford_raw"
master_dir = "./master_dataset"

if not os.path.exists(master_dir):
    print("📥 Extracting and compiling Oxford 12-Universe...")
    urllib.request.urlretrieve(url, archive_name)
    os.makedirs(extraction_dir, exist_ok=True)
    with tarfile.open(archive_name, "r:gz") as tar:
        tar.extractall(path=extraction_dir)

    # Decoupling 12 specific classes out of the sequential image bundle
    class_slices = {
        "Bluebell": (880, 960),
        "Buttercup": (640, 720),
        "Coltsfoot": (320, 400),
        "Cowslip": (400, 480),
        "Crocus": (160, 240),
        "Daffodil": (0, 80),
        "Daisy": (720, 800),
        "Fritillary": (560, 640),
        "Iris": (240, 320),
        "Lily": (80, 160),
        "Pansy": (1200, 1280),
        "Sunflower": (480, 560),
    }

    all_images = sorted(os.listdir(f"{extraction_dir}/jpg"))
    for class_name, (start, end) in class_slices.items():
        class_folder = f"{master_dir}/{class_name}"
        os.makedirs(class_folder, exist_ok=True)
        for idx in range(start, end):
            src_path = f"{extraction_dir}/jpg/{all_images[idx]}"
            dst_path = f"{class_folder}/{all_images[idx]}"
            os.replace(src_path, dst_path)

    import shutil

    shutil.rmtree(extraction_dir)
    os.remove(archive_name)
    print("✅ Complete structural dataset setup ready.")

In [5]:
# 3. Form Dataset & Track Explicit Class Mappings
dataset = datasets.ImageFolder(root=master_dir, transform=transform)
class_list = sorted(os.listdir(master_dir))
class_mapping = {name: idx for idx, name in enumerate(class_list)}

for i in range(len(dataset.samples)):
    path, _ = dataset.samples[i]
    folder_name = os.path.basename(os.path.dirname(path))
    dataset.samples[i] = (path, class_mapping[folder_name])

# 4. Partition Split (80% Train, 20% Evaluation Validation)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)

In [6]:
print("🧠 Initializing Pre-trained MobileNetV2 Engine Layers...")
weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)

# Freeze base features to retain foundational edge/shape pattern weights
for param in model.parameters():
    param.requires_grad = False

# Swap the final classification head with our targeted 12-Universe output layer
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 12)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.005)

🧠 Initializing Pre-trained MobileNetV2 Engine Layers...


In [7]:
# 6. Fine-Tuning Execution
print("\n--- Initiating Fine-Tuning Run ---")
start_time = time.time()
for epoch in range(5):  # Fewer epochs required due to pre-trained base
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(
        f"Epoch [{epoch+1}/5] Completed -> Batch Mean Loss: {running_loss/len(train_loader):.4f}"
    )


--- Initiating Fine-Tuning Run ---
Epoch [1/5] Completed -> Batch Mean Loss: 1.2538
Epoch [2/5] Completed -> Batch Mean Loss: 0.3776
Epoch [3/5] Completed -> Batch Mean Loss: 0.2405
Epoch [4/5] Completed -> Batch Mean Loss: 0.1822
Epoch [5/5] Completed -> Batch Mean Loss: 0.1338


In [8]:
# 7. Validation Evaluation Test
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\n🌟 Target Model Fine-Tuning Accuracy: {(correct/total)*100:.2f}%\n")


🌟 Target Model Fine-Tuning Accuracy: 91.67%



In [9]:
# 8. Optimized Web ONNX Graph Compilation
model.to("cpu")
model.eval()
dummy_input = torch.randn(1, 3, 224, 224)  # Adjusted to standard 224 resolution
torch.onnx.export(
    model,
    dummy_input,
    "../web/flower_model.onnx",
    input_names=["input"],
    output_names=["output"],
    export_params=True,
    external_data=False,
)
print("📦 Serialized Model Asset saved to '../web/flower_model.onnx'")

[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\Shreya\AppData\Local\Programs\Python\Python313\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
📦 Serialized Model Asset saved to '../web/flower_model.onnx'


In [10]:
# 9. Dynamic Generation of Target Array Key Configuration File
labels_dict = {str(idx): name for idx, name in enumerate(class_list)}
with open("../web/labels.json", "w") as f:
    json.dump(labels_dict, f, indent=4)
print("📦 Mapping asset configuration written to '../web/labels.json'")

📦 Mapping asset configuration written to '../web/labels.json'
